## Why this analysis

Mosul is Iraq's second-largest city and the historical center of Nineveh Governorate. From 2014 to 2017 the city was occupied by ISIS; the battle for liberation devastated wide areas, particularly the Old City on the west bank of the Tigris. Since 2017, reconstruction has been uneven. The east bank, less damaged, has largely recovered, while the western neighborhoods continue to rebuild block by block.

The atlas question is the same as in Cox's Bazar and Kabul: where does the population live, where are the health facilities, and what is the spatial gap between them?

In [ ]:
# Mosul — Service Density Analysis
# Part of the Atlas of Urban Fragility

import osmnx as ox
import geopandas as gpd
import pandas as pd
import folium
import h3
from shapely.geometry import Polygon, Point
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Sanity check
print(f"osmnx:      {ox.__version__}")
print(f"geopandas:  {gpd.__version__}")
print(f"folium:     {folium.__version__}")
print(f"h3:         {h3.__version__}")

# Project paths
PROJECT_ROOT = Path.cwd().parent  # we're in atlas/, so parent is heissj.github.io/
DATA_DIR = PROJECT_ROOT / "data" / "mosul"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"\nData directory: {DATA_DIR}")

# Verify we're on the right Python
import sys
print(f"\nPython: {sys.executable}")

In [ ]:
# Area of interest: central Mosul
# Covers central districts on both banks of the Tigris, including the Old City
AREA = {
    'name': "Mosul (central districts)",
    'bbox': {
        'south': 36.30,
        'north': 36.40,
        'west':  43.10,
        'east':  43.20,
    }
}

center_lat = (AREA['bbox']['south'] + AREA['bbox']['north']) / 2
center_lon = (AREA['bbox']['west']  + AREA['bbox']['east'])  / 2

print(f"Area:   {AREA['name']}")
print(f"Center: {center_lat:.4f}°N, {center_lon:.4f}°E")
print(f"Span:   {AREA['bbox']['north']-AREA['bbox']['south']:.3f}° lat × "
      f"{AREA['bbox']['east']-AREA['bbox']['west']:.3f}° lon")

In [ ]:
# Quick map to confirm the area looks right
m = folium.Map(location=[center_lat, center_lon], zoom_start=12,
               tiles='OpenStreetMap')

# Draw the bounding box
b = AREA['bbox']
folium.Rectangle(
    bounds=[[b['south'], b['west']], [b['north'], b['east']]],
    color='#cc0000', weight=2,
    fill=True, fill_opacity=0.1,
    tooltip=AREA['name']
).add_to(m)

m

## Where are the health facilities?

OpenStreetMap coverage of Mosul has improved substantially since 2017, with humanitarian mapping efforts documenting post-war infrastructure. Facility data still has gaps, especially for smaller or newer clinics, but major hospitals and pharmacies in the central districts are reasonably represented. This query captures hospitals, clinics, doctors' offices, pharmacies, dentists, and anything tagged as healthcare.

In [ ]:
# OSM tags for health facilities — covers hospitals, clinics, pharmacies,
# doctors' offices, and anything tagged as healthcare
tags = {
    'amenity': ['hospital', 'clinic', 'doctors', 'pharmacy'],
    'healthcare': True,
}

# osmnx 2.x bbox format: (west, south, east, north)
bbox_tuple = (b['west'], b['south'], b['east'], b['north'])

print(f"Querying OpenStreetMap for health facilities in {AREA['name']}...")
facilities = ox.features.features_from_bbox(bbox=bbox_tuple, tags=tags)
print(f"\nReturned {len(facilities)} features")
print(f"\nFirst few rows:")
facilities.head()

In [ ]:
# What kinds of facilities are in the data?
print("Amenity types:")
print(facilities['amenity'].value_counts() if 'amenity' in facilities.columns else "(no amenity column)")

print("\nHealthcare types:")
print(facilities['healthcare'].value_counts() if 'healthcare' in facilities.columns else "(no healthcare column)")

print(f"\nGeometry types:")
print(facilities.geometry.geom_type.value_counts())

In [ ]:
# Convert polygon geometries to their centroids; points stay as points
# Project to Web Mercator first for accurate centroid math, then back to WGS84
facilities_proj = facilities.to_crs(epsg=3857)
facilities_proj['geometry'] = facilities_proj.geometry.centroid
facilities_pts = facilities_proj.to_crs(epsg=4326)

print(f"{len(facilities_pts)} facilities, all as points now")
print(f"Geometry types: {facilities_pts.geometry.geom_type.value_counts().to_dict()}")

In [ ]:
# H3 resolution 8 → hexagons roughly 0.5 km across
# (Good resolution for a camp-scale analysis; res 7 = too coarse, res 9 = too fine)
H3_RESOLUTION = 8

# Define our area as an H3 polygon (vertices in lat, lng order)
vertices = [
    (b['south'], b['west']),
    (b['north'], b['west']),
    (b['north'], b['east']),
    (b['south'], b['east']),
]
h3_poly = h3.LatLngPoly(vertices)

# Get all H3 cells inside this polygon
hex_ids = h3.h3shape_to_cells(h3_poly, H3_RESOLUTION)
print(f"Generated {len(hex_ids)} H3 hexes at resolution {H3_RESOLUTION}")
print(f"First few hex IDs: {hex_ids[:3]}")

In [ ]:
# Build a GeoDataFrame where each row is one hexagon
hex_records = []
for h in hex_ids:
    # h3.cell_to_boundary returns list of (lat, lng) tuples
    # Shapely expects (lng, lat) order
    boundary = h3.cell_to_boundary(h)
    poly = Polygon([(lng, lat) for lat, lng in boundary])
    hex_records.append({'h3_id': h, 'geometry': poly})

hexes = gpd.GeoDataFrame(hex_records, crs='EPSG:4326')

# Spatial join: which hex contains each facility?
joined = gpd.sjoin(facilities_pts, hexes, how='left', predicate='within')

# Count facilities per hex
counts = joined.groupby('h3_id').size().rename('facility_count')
hexes = hexes.merge(counts, on='h3_id', how='left')
hexes['facility_count'] = hexes['facility_count'].fillna(0).astype(int)

# Quick stats
print(f"Total hexes:                {len(hexes)}")
print(f"Hexes with ≥1 facility:     {(hexes['facility_count'] > 0).sum()}")
print(f"Hexes with ≥3 facilities:   {(hexes['facility_count'] >= 3).sum()}")
print(f"Max facilities in one hex:  {hexes['facility_count'].max()}")
print(f"\nDistribution of facility counts:")
print(hexes['facility_count'].value_counts().sort_index())

In [ ]:
# Set up the map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles='OpenStreetMap'
)

# Style function: color hexes by facility count
def style_fn(feature):
    count = feature['properties']['facility_count']
    if count == 0:
        return {'fillColor': '#ffffff', 'color': '#cccccc',
                'weight': 0.3, 'fillOpacity': 0.0}
    elif count == 1:
        return {'fillColor': '#9ecae1', 'color': '#4292c6',
                'weight': 0.6, 'fillOpacity': 0.6}
    elif count == 2:
        return {'fillColor': '#4292c6', 'color': '#2171b5',
                'weight': 0.6, 'fillOpacity': 0.7}
    else:  # 3+
        return {'fillColor': '#08519c', 'color': '#08306b',
                'weight': 0.8, 'fillOpacity': 0.85}

# Add the hexes
folium.GeoJson(
    hexes.to_json(),
    name='Health facility density (per hex)',
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(
        fields=['facility_count'],
        aliases=['Health facilities:']
    )
).add_to(m)

# Add individual facility points on top for context
for _, row in facilities_pts.iterrows():
    name = (row.get('name:en') or row.get('name')
            or str(row.get('amenity', 'Health facility')))
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3, color='#222', weight=1,
        fill=True, fillColor='white', fillOpacity=1.0,
        tooltip=name
    ).add_to(m)

# Add a simple legend (fixed-position HTML overlay)
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px;
            background-color: white; border: 1px solid #999;
            padding: 10px 14px; font-family: Arial, sans-serif;
            font-size: 12px; z-index: 9999;
            box-shadow: 0 2px 6px rgba(0,0,0,0.15); line-height: 1.6;">
    <div style="font-weight: bold; margin-bottom: 6px;">
        Health facilities per hex
    </div>
    <div><span style="background:#08519c;opacity:0.85;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>3 or more</div>
    <div><span style="background:#4292c6;opacity:0.7;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>2</div>
    <div><span style="background:#9ecae1;opacity:0.6;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>1</div>
    <div><span style="background:white;border:1px solid #ccc;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>0</div>
    <div style="border-top:1px solid #eee;margin-top:6px;padding-top:4px;font-size:11px;color:#666;">
        Hex resolution: H3 res 8 (~0.5 km)
    </div>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Display
m

In [ ]:
output_path = PROJECT_ROOT / "atlas" / "mosul-map.html"
m.save(str(output_path))
print(f"Map saved to: {output_path}")

In [ ]:
# Buildings as a population proxy
# (Where structures cluster, people cluster)

print(f"Querying OpenStreetMap for buildings in {AREA['name']}...")
buildings = ox.features.features_from_bbox(
    bbox=bbox_tuple,
    tags={'building': True}
)
print(f"\nReturned {len(buildings):,} building features")
print(f"Geometry types: {buildings.geometry.geom_type.value_counts().to_dict()}")

In [ ]:
# Convert to points (centroid)
buildings_proj = buildings.to_crs(epsg=3857)
buildings_proj['geometry'] = buildings_proj.geometry.centroid
buildings_pts = buildings_proj.to_crs(epsg=4326)

print(f"{len(buildings_pts):,} buildings, now as point geometries")

In [ ]:
# Spatial join: which hex contains each building?
b_joined = gpd.sjoin(buildings_pts[['geometry']], hexes, how='left', predicate='within')

# Count buildings per hex
b_counts = b_joined.groupby('h3_id').size().rename('building_count')

# Merge into our hex GeoDataFrame
hexes = hexes.merge(b_counts, on='h3_id', how='left')
hexes['building_count'] = hexes['building_count'].fillna(0).astype(int)

# Stats
print(f"Total hexes:                       {len(hexes)}")
print(f"Hexes with ≥1 building:            {(hexes['building_count'] > 0).sum()}")
print(f"Hexes with ≥50 buildings:          {(hexes['building_count'] >= 50).sum()}")
print(f"Hexes with ≥200 buildings:         {(hexes['building_count'] >= 200).sum()}")
print(f"Max buildings in a single hex:     {hexes['building_count'].max():,}")
print(f"Median building count (where >0):  {int(hexes[hexes['building_count']>0]['building_count'].median())}")

In [ ]:
# The portfolio money shot:
# How many hexes have lots of buildings but zero facilities?

dense_no_service = hexes[(hexes['building_count'] >= 50) & (hexes['facility_count'] == 0)]
dense_with_service = hexes[(hexes['building_count'] >= 50) & (hexes['facility_count'] > 0)]

print(f"Hexes with ≥50 buildings AND 0 facilities:  {len(dense_no_service)}")
print(f"Hexes with ≥50 buildings AND ≥1 facility:   {len(dense_with_service)}")
print(f"\nIn other words: of all the densely-populated areas in this bbox,")
print(f"{len(dense_no_service)} out of {len(dense_no_service)+len(dense_with_service)} have no health facility nearby.")

## The gap

Building footprints as a population proxy require some interpretation in Mosul. OSM data reflects what is currently mapped, which can include standing structures and possibly some destroyed during the conflict that remain in the OSM record. Despite that caveat, the metric still tells us where built fabric is densest, approximating where residents have returned and rebuilt.

We classify each 0.5 km hex into one of three categories:

- **Dense, no facility** — building count ≥ 50, facility count = 0. The gap zones.
- **Dense, served** — building count ≥ 50, facility count ≥ 1. Adequately reached.
- **Low density** — fewer than 50 mapped buildings. Outside the dense urban footprint, including some of the most heavily damaged Old City blocks.

In [ ]:
# Classify each hex into one of three categories
def classify_hex(row):
    if row['building_count'] < 50:
        return 'low_density'        # rural / outside camp / empty
    elif row['facility_count'] == 0:
        return 'underserved'        # dense but no facility
    else:
        return 'served'             # dense and at least 1 facility

hexes['access_class'] = hexes.apply(classify_hex, axis=1)

# Summary
print("Hexes by access class:")
print(hexes['access_class'].value_counts())

print(f"\nBuildings in 'underserved' hexes:  "
      f"{hexes[hexes['access_class']=='underserved']['building_count'].sum():>7,}")
print(f"Buildings in 'served' hexes:       "
      f"{hexes[hexes['access_class']=='served']['building_count'].sum():>7,}")

# The headline number
total_dense = hexes[hexes['building_count'] >= 50]['building_count'].sum()
underserved = hexes[hexes['access_class']=='underserved']['building_count'].sum()
pct = 100 * underserved / total_dense
print(f"\n{pct:.1f}% of buildings in dense areas are in 'underserved' hexes.")

In [ ]:
# Service-population alignment map
m3 = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles='OpenStreetMap'
)

# Color by access class — red where the gap is
def gap_style_fn(feature):
    cls = feature['properties']['access_class']
    if cls == 'low_density':
        return {'fillColor': 'transparent', 'color': '#cccccc',
                'weight': 0.3, 'fillOpacity': 0.0}
    elif cls == 'underserved':
        return {'fillColor': '#cb181d', 'color': '#67000d',
                'weight': 0.6, 'fillOpacity': 0.65}
    else:  # served
        return {'fillColor': '#41ab5d', 'color': '#005a32',
                'weight': 0.6, 'fillOpacity': 0.55}

folium.GeoJson(
    hexes.to_json(),
    style_function=gap_style_fn,
    tooltip=folium.GeoJsonTooltip(
        fields=['building_count', 'facility_count', 'access_class'],
        aliases=['Buildings:', 'Health facilities:', 'Status:']
    )
).add_to(m3)

# Facility points on top
for _, row in facilities_pts.iterrows():
    name = (row.get('name:en') or row.get('name')
            or str(row.get('amenity', 'Health facility')))
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4, color='#222', weight=1.5,
        fill=True, fillColor='white', fillOpacity=1.0,
        tooltip=name
    ).add_to(m3)

# Legend
gap_legend = '''
<div style="position: fixed; bottom: 30px; left: 30px;
            background-color: white; border: 1px solid #999;
            padding: 10px 14px; font-family: Arial, sans-serif;
            font-size: 12px; z-index: 9999;
            box-shadow: 0 2px 6px rgba(0,0,0,0.15); line-height: 1.6;
            max-width: 240px;">
    <div style="font-weight: bold; margin-bottom: 6px;">
        Service-population alignment
    </div>
    <div><span style="background:#cb181d;opacity:0.65;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>
         Dense, no health facility</div>
    <div><span style="background:#41ab5d;opacity:0.55;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>
         Dense, served by 1+ facility</div>
    <div><span style="background:white;border:1px solid #ccc;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>
         Low density (&lt;50 buildings)</div>
    <div style="border-top:1px solid #eee;margin-top:6px;padding-top:4px;font-size:11px;color:#666;">
        Hex resolution: H3 res 8 (~0.5 km)<br>
        Buildings used as population proxy<br>
        Sources: OpenStreetMap, HOTOSM
    </div>
</div>
'''
m3.get_root().html.add_child(folium.Element(gap_legend))

# Save it
gap_path = PROJECT_ROOT / "atlas" / "mosul-gap-map.html"
m3.save(str(gap_path))
print(f"Gap map saved to: {gap_path}")

m3

## What the map shows

Mosul's central districts straddle the Tigris River. Historically the east bank suffered less damage during the 2014-2017 conflict and recovered faster, while the west bank, especially the Old City, was devastated and continues to rebuild slowly. The gap map highlights where dense building footprints persist without nearby health facilities. Hexes near the western bank deserve particular attention given the slower reconstruction pace there.

Reading the map alongside Cox's Bazar and Kabul reveals how different fragility types produce different spatial patterns. In Cox's Bazar, services concentrated in the camp's population core; in Kabul, services formed scattered clusters in a sprawling capital. In Mosul, the analytical question becomes whether the river itself functions as a service boundary.

### Limitations

- **OSM facility data is incomplete.** Iraq's healthcare system includes many private clinics that may not appear in OSM. Public-sector and donor-supported facilities are better represented.
- **Building footprints may overstate occupancy.** Some buildings may be vacant, destroyed, or still in repair. Without ground-truth occupancy data, building density is a rough proxy for residency.
- **Conflict damage is invisible here.** A natural next layer would be UNOSAT damage assessments to compare gap zones against destroyed blocks.
- **Presence is not capacity.** A small primary clinic and a regional hospital both count as one facility. The map shows location, not adequacy.

### Methodology

OpenStreetMap features pulled via `osmnx`. Hexagonal tiling via Uber's H3 (resolution 8). Spatial joins and aggregation in `geopandas`. Maps rendered with `folium`. All code is visible in the cells above (click "Show code" on any block).